# RoadScan - YOLO11 road-damage training on RDD2022

**Just upload this notebook to Kaggle and run all cells.**

**Before running (Kaggle right-hand panel -> Session options):**
1. **Accelerator:** GPU T4 x2 (or P100). Training uses one GPU.
2. **Internet:** ON (needed to download the dataset and the YOLO weights).
3. *(Optional, no download needed)* Add Input -> search "RDD2022" -> attach it. It is auto-detected.

**What this notebook does:** clean up -> get RDD2022 (verified download) -> convert to YOLO format with a real 85/15 train/val split -> train -> evaluate -> export `best.pt` + ONNX -> `results.zip`.

**Disk layout (why earlier runs ran out of space):** the 13 GB zip goes to `/kaggle/working`, the extracted/converted data goes to the temp disk, and only models/results stay in `/kaggle/working`. Test images (unlabelled) are skipped.

**Time:** the full dataset is ~30k+ labelled images. `MAX_HOURS = 9` in the config cell stops training safely inside Kaggle's 12 h limit. Lower it (or `EPOCHS`) for a quicker run.

**Outputs:** after the run, click *Save Version* -> *Save & Run All*, then download `roadscan/results.zip` from the Output tab.

In [1]:
# Step 0 - free disk space from any earlier failed run (keeps the downloaded zip, so no re-download)
import shutil
for p in ['/kaggle/working/roadscan/raw', '/kaggle/working/roadscan/yolo', '/kaggle/working/roadscan/temp',
          '/kaggle/working/roadscan/models']:
    shutil.rmtree(p, ignore_errors=True)
print('Free on /kaggle/working: %.1f GB' % (shutil.disk_usage('/kaggle/working').free / 1e9))

Free on /kaggle/working: 20.9 GB


In [2]:
!pip install -q -U ultralytics

In [3]:
import os, json, shutil, random, zipfile
from pathlib import Path
from collections import Counter
import torch

print(f'CUDA available: {torch.cuda.is_available()}')

# ----------------------------- CONFIG -----------------------------
IS_KAGGLE = Path('/kaggle/working').exists()
BASE_DIR  = Path('/kaggle/working') if IS_KAGGLE else Path.cwd()

# /kaggle/working is only ~20 GB, but the RDD2022 zip alone is 13 GB. So:
#   - the zip is downloaded to the working disk (deleted right after extraction)
#   - extracted + converted data go to the temp disk (/kaggle/temp, falls back to /tmp)
#   - only models / results are kept in /kaggle/working
WORK_DIR   = BASE_DIR / 'roadscan'
DL_DIR     = WORK_DIR / 'download'
MODELS_DIR = WORK_DIR / 'models'

if IS_KAGGLE:
    # pick whichever scratch disk has the most free space
    _cands = []
    for _c in (Path('/kaggle/temp'), Path('/tmp')):
        try:
            _c.mkdir(parents=True, exist_ok=True)
            _cands.append((shutil.disk_usage(_c).free, str(_c)))
        except OSError:
            pass
    TEMP_BASE = Path(max(_cands)[1]) if _cands else Path('/kaggle/working')
else:
    TEMP_BASE = BASE_DIR / 'tmp'
    TEMP_BASE.mkdir(parents=True, exist_ok=True)
RAW_DIR  = TEMP_BASE / 'roadscan_raw'    # extracted dataset (only used when downloading)
YOLO_DIR = TEMP_BASE / 'roadscan_yolo'   # converted YOLO dataset (same disk as RAW_DIR -> hard links)

# clean up leftovers from earlier failed runs (they can fill the disk)
for leftover in [WORK_DIR / 'raw', WORK_DIR / 'yolo', WORK_DIR / 'temp', RAW_DIR, YOLO_DIR]:
    shutil.rmtree(leftover, ignore_errors=True)

# Option A: path of an RDD2022 dataset you attached on Kaggle, e.g. '/kaggle/input/rdd2022'
#           Leave None to auto-detect under /kaggle/input, else it will be downloaded.
KAGGLE_INPUT_DIR = None
FIGSHARE_ARTICLE_ID = 21431547

KEEP_CLASSES = {'D00': 0, 'D10': 1, 'D20': 2, 'D40': 3}
CLASS_NAMES  = list(KEEP_CLASSES.keys())

VAL_SPLIT = 0.15
SEED      = 42

MODEL     = 'yolo11s.pt'
EPOCHS    = 50
IMGSZ     = 640
BATCH     = 16
PATIENCE  = 10
MAX_HOURS = 9.0    # stop training after this many hours (Kaggle caps sessions at ~12h). None = use EPOCHS only.
# -------------------------------------------------------------------

for d in [WORK_DIR, MODELS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f'Free space  working: {shutil.disk_usage(WORK_DIR).free/1e9:.1f} GB | '
      f'temp ({TEMP_BASE}): {shutil.disk_usage(TEMP_BASE).free/1e9:.1f} GB')

CUDA available: True
Free space  working: 20.9 GB | temp (/tmp): 1038.2 GB


## 1. Get RDD2022

In [4]:
def has_voc_annotations(root):
    for _, _, files in os.walk(root):
        if any(f.lower().endswith('.xml') for f in files):
            return True
    return False


def download_figshare(article_id, dest_dir):
    import requests
    from tqdm import tqdm
    dest_dir.mkdir(parents=True, exist_ok=True)
    try:
        meta = requests.get(f'https://api.figshare.com/v2/articles/{article_id}', timeout=60)
        meta.raise_for_status()
    except requests.RequestException as e:
        raise RuntimeError(
            'Could not reach figshare. On Kaggle, enable Internet in the notebook settings '
            f'(or attach RDD2022 via Add Data instead). Original error: {e}')
    files = meta.json()['files']
    print('Files in article:', [(f['name'], round(f['size'] / 1e9, 2)) for f in files], '(name, GB)')
    for f in files:
        out = dest_dir / f['name']
        if out.exists() and out.stat().st_size == f['size']:
            print(f"  already downloaded: {f['name']}")
            continue
        with requests.get(f['download_url'], stream=True, timeout=60, allow_redirects=True) as r:
            r.raise_for_status()
            with open(out, 'wb') as fh, tqdm(total=f['size'], unit='B', unit_scale=True, desc=f['name']) as bar:
                for chunk in r.iter_content(1 << 20):
                    fh.write(chunk)
                    bar.update(len(chunk))
        got = out.stat().st_size
        if got != f['size']:
            raise RuntimeError(f"Incomplete download for {f['name']}: {got} of {f['size']} bytes")


def extract_all_zips(src_dir, dest_dir, delete_zips=True, skip_dirs=('test', 'tests')):
    # Extract every zip under src_dir (and zips nested inside) into dest_dir.
    # Members inside a 'test' folder are skipped: RDD2022 test images have no annotations,
    # so they are useless for training and would waste ~20% of the disk.
    dest_dir.mkdir(parents=True, exist_ok=True)
    pending = sorted(Path(src_dir).rglob('*.zip'))
    while pending:
        for zf in pending:
            with zipfile.ZipFile(zf) as z:
                members = [m for m in z.infolist()
                           if not m.is_dir()
                           and not any(p.lower() in skip_dirs for p in Path(m.filename).parts)
                           and '__MACOSX' not in m.filename]
                need = sum(m.file_size for m in members)
                free = shutil.disk_usage(dest_dir).free
                print(f'  extracting {zf.name}: {len(members)} files, {need/1e9:.1f} GB '
                      f'(free: {free/1e9:.1f} GB)')
                if need * 1.02 > free:
                    raise RuntimeError(
                        f'Not enough disk space on {dest_dir}: need {need/1e9:.1f} GB, '
                        f'have {free/1e9:.1f} GB. Attach RDD2022 as a Kaggle dataset instead.')
                z.extractall(dest_dir, members=members)
            if delete_zips:
                zf.unlink()
        pending = sorted(dest_dir.rglob('*.zip'))   # nested zips


DOWNLOADED = False
if KAGGLE_INPUT_DIR:
    RAW_DIR = Path(KAGGLE_INPUT_DIR)
elif Path('/kaggle/input').exists() and has_voc_annotations('/kaggle/input'):
    RAW_DIR = Path('/kaggle/input')
else:
    print('No attached dataset found -> downloading from figshare')
    download_figshare(FIGSHARE_ARTICLE_ID, DL_DIR)
    print('Extracting...')
    extract_all_zips(DL_DIR, RAW_DIR)
    shutil.rmtree(DL_DIR, ignore_errors=True)
    DOWNLOADED = True

print('RAW_DIR =', RAW_DIR)
assert RAW_DIR.exists() and has_voc_annotations(RAW_DIR), (
    f'No Pascal-VOC .xml annotations found under {RAW_DIR}. '
    'Check the dataset path / download.')

No attached dataset found -> downloading from figshare
Files in article: [('RDD2022_released_through_CRDDC2022.zip', 13.26), ('Directory_Structure_CRDDC_RDD2022.txt', 0.0), ('File_List_CRDDC_RDD2022.txt', 0.0), ('label_map.pbtxt', 0.0)] (name, GB)


RDD2022_released_through_CRDDC2022.zip: 100%|██████████| 13.3G/13.3G [03:37<00:00, 61.0MB/s]
Directory_Structure_CRDDC_RDD2022.txt: 100%|██████████| 1.16k/1.16k [00:00<00:00, 2.88MB/s]
File_List_CRDDC_RDD2022.txt: 100%|██████████| 3.25M/3.25M [00:00<00:00, 38.4MB/s]
label_map.pbtxt: 100%|██████████| 127/127 [00:00<00:00, 574kB/s]


Extracting...
  extracting RDD2022_released_through_CRDDC2022.zip: 7 files, 13.3 GB (free: 1038.2 GB)
  extracting China_Drone.zip: 4802 files, 0.2 GB (free: 1024.9 GB)
  extracting China_MotorBike.zip: 3954 files, 0.2 GB (free: 1024.9 GB)
  extracting Czech.zip: 5658 files, 0.2 GB (free: 1024.9 GB)
  extracting India.zip: 15412 files, 0.5 GB (free: 1025.0 GB)
  extracting Japan.zip: 21012 files, 0.9 GB (free: 1024.9 GB)
  extracting Norway.zip: 16322 files, 8.8 GB (free: 1025.1 GB)
  extracting United_States.zip: 9610 files, 0.4 GB (free: 1026.8 GB)
RAW_DIR = /tmp/roadscan_raw


## 2. Convert to YOLO (with train/val split)

In [5]:
import xml.etree.ElementTree as ET
from PIL import Image
from tqdm import tqdm

IMG_EXTS = {'.jpg', '.jpeg', '.png'}


def voc_to_yolo(xml_path, w, h):
    try:
        root = ET.parse(xml_path).getroot()
    except (ET.ParseError, OSError):
        return []
    lines = []
    for obj in root.findall('object'):
        cls = (obj.findtext('name') or '').strip()
        if cls not in KEEP_CLASSES:
            continue
        bb = obj.find('bndbox')
        if bb is None:
            continue
        try:
            x1 = float(bb.findtext('xmin', '0')); y1 = float(bb.findtext('ymin', '0'))
            x2 = float(bb.findtext('xmax', '0')); y2 = float(bb.findtext('ymax', '0'))
        except ValueError:
            continue
        # clip to image and drop degenerate boxes
        x1, x2 = max(0.0, min(x1, w)), max(0.0, min(x2, w))
        y1, y2 = max(0.0, min(y1, h)), max(0.0, min(y2, h))
        if x2 - x1 < 1 or y2 - y1 < 1:
            continue
        xc, yc = (x1 + x2) / 2 / w, (y1 + y2) / 2 / h
        bw, bh = (x2 - x1) / w, (y2 - y1) / h
        lines.append(f'{KEEP_CLASSES[cls]} {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}')
    return lines


def collect_samples(raw_root):
    # Find every folder containing .xml files that has a sibling images folder.
    # Works for Country/train/annotations/xmls + Country/train/images and for VOC layouts.
    samples, per_split = [], Counter()
    for dirpath, _, files in os.walk(raw_root):
        if not any(f.lower().endswith('.xml') for f in files):
            continue
        xml_dir = Path(dirpath)
        candidates = [xml_dir.parent.parent / 'images', xml_dir.parent / 'images',
                      xml_dir.parent.parent / 'JPEGImages', xml_dir.parent / 'JPEGImages',
                      xml_dir]
        img_dir = next((c for c in candidates if c.is_dir() and
                        any(p.suffix.lower() in IMG_EXTS for p in c.iterdir())), None)
        if img_dir is None:
            continue
        try:
            tag = '_'.join(img_dir.parent.relative_to(raw_root).parts) or 'root'
        except ValueError:
            tag = 'root'
        images = {p.stem: p for p in img_dir.iterdir() if p.suffix.lower() in IMG_EXTS}
        for xml_file in xml_dir.glob('*.xml'):
            img = images.get(xml_file.stem)
            if img is not None:
                samples.append((tag, img, xml_file))
                per_split[tag] += 1
    return samples, per_split


def place(src, dst):
    # hard-link when on the same filesystem (no extra disk), otherwise copy
    if dst.exists():
        return
    try:
        os.link(src, dst)
    except OSError:
        shutil.copy2(src, dst)


samples, per_split = collect_samples(RAW_DIR)
print('Annotated image/xml pairs found per folder:')
for k, v in sorted(per_split.items()):
    print(f'  {k}: {v}')
assert samples, 'No image/annotation pairs found - check folder layout of the dataset.'

# fresh output dir (avoids stale files from earlier failed runs)
if YOLO_DIR.exists():
    shutil.rmtree(YOLO_DIR)
for s in ('train', 'val'):
    (YOLO_DIR / 'images' / s).mkdir(parents=True, exist_ok=True)
    (YOLO_DIR / 'labels' / s).mkdir(parents=True, exist_ok=True)

random.Random(SEED).shuffle(samples)
n_val = int(len(samples) * VAL_SPLIT)
splits = {'val': samples[:n_val], 'train': samples[n_val:]}

stats = {s: Counter() for s in splits}
n_done = {s: 0 for s in splits}
for split, items in splits.items():
    for tag, img_path, xml_path in tqdm(items, desc=split):
        try:
            with Image.open(img_path) as im:
                w, h = im.size
        except (OSError, ValueError):
            continue
        lines = voc_to_yolo(xml_path, w, h)
        stem = f'{tag}_{img_path.stem}'
        place(img_path, YOLO_DIR / 'images' / split / (stem + img_path.suffix.lower()))
        (YOLO_DIR / 'labels' / split / (stem + '.txt')).write_text('\n'.join(lines))
        n_done[split] += 1
        for ln in lines:
            stats[split][CLASS_NAMES[int(ln.split()[0])]] += 1

print('\nImages:', n_done)
for s in stats:
    print(f'{s} instances per class:', dict(stats[s]))
assert n_done['train'] > 0 and n_done['val'] > 0, 'Conversion produced no images.'

if DOWNLOADED:                       # free disk space (hard links keep the converted copies)
    shutil.rmtree(RAW_DIR, ignore_errors=True)

Annotated image/xml pairs found per folder:
  China_Drone_train: 2401
  China_MotorBike_train: 1977
  Czech_train: 2829
  India_train: 7706
  Japan_train: 10506
  Norway_train: 8161
  United_States_train: 4805


train: 100%|██████████| 32628/32628 [00:35<00:00, 910.76it/s] 



Images: {'val': 5757, 'train': 32628}
val instances per class: {'D00': 4083, 'D20': 1561, 'D10': 1832, 'D40': 971}
train instances per class: {'D10': 9998, 'D20': 9055, 'D00': 21933, 'D40': 5573}


In [6]:
import yaml
data_cfg = {
    'path':  str(YOLO_DIR),
    'train': 'images/train',
    'val':   'images/val',
    'nc':    len(CLASS_NAMES),
    'names': {i: n for i, n in enumerate(CLASS_NAMES)},
}
DATA_YAML = YOLO_DIR / 'data.yaml'
DATA_YAML.write_text(yaml.safe_dump(data_cfg, sort_keys=False))
print(DATA_YAML.read_text())

path: /tmp/roadscan_yolo
train: images/train
val: images/val
nc: 4
names:
  0: D00
  1: D10
  2: D20
  3: D40



## 3. Train YOLO11

In [7]:
from ultralytics import YOLO

model = YOLO(MODEL)
train_args = dict(
    data=str(DATA_YAML), epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH,
    device=0 if torch.cuda.is_available() else 'cpu',
    project=str(MODELS_DIR), name='roadscan', exist_ok=True,
    patience=PATIENCE, workers=2, seed=SEED, plots=True,
)
if MAX_HOURS:
    train_args['time'] = MAX_HOURS       # hours; overrides epochs

results = model.train(**train_args)
SAVE_DIR = Path(model.trainer.save_dir)
BEST_PT = SAVE_DIR / 'weights' / 'best.pt'
print('Best weights:', BEST_PT, BEST_PT.exists())

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
Ultralytics 8.4.204 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/tmp/roadscan_yolo/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, fr

/usr/local/lib/python3.13/dist-packages/ray/train/_internal/session.py:683: UserWarning: `get_trial_id` is meant to only be called inside a function that is executed by a Tuner or Trainer. Returning `None`.
  warnings.warn(


       2/52      5.08G      2.076       2.52      1.768          7        640: 100% ━━━━━━━━━━━━ 2040/2040 3.9it/s 8:38
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 180/180 2.7it/s 1:07
                   all       5757       8447      0.375      0.226      0.179     0.0738

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       3/54      5.08G      2.149       2.65      1.864         17        640: 100% ━━━━━━━━━━━━ 2040/2040 4.1it/s 8:21
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 180/180 2.7it/s 1:08
                   all       5757       8447      0.301      0.231      0.176     0.0696

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       4/55      5.08G      2.127      2.629      1.881          7        640: 100% ━━━━━━━━━━━━ 2040/2040 4.1it/s 8:14
                 Class     Images  Instances  

## 4. Evaluate (best checkpoint, validation split)

In [8]:
best_model = YOLO(str(BEST_PT))
metrics = best_model.val(data=str(DATA_YAML), split='val', imgsz=IMGSZ)

print(f'mAP50:     {metrics.box.map50:.4f}')
print(f'mAP50-95:  {metrics.box.map:.4f}')
print(f'Precision: {metrics.box.mp:.4f}')
print(f'Recall:    {metrics.box.mr:.4f}')
per_class = {}
for i, c in enumerate(metrics.box.ap_class_index):
    p, r, ap50, ap = metrics.box.class_result(i)
    per_class[CLASS_NAMES[int(c)]] = {'precision': float(p), 'recall': float(r),
                                      'mAP50': float(ap50), 'mAP50-95': float(ap)}
    print(f'  {CLASS_NAMES[int(c)]}: P={p:.3f} R={r:.3f} mAP50={ap50:.3f} mAP50-95={ap:.3f}')

Ultralytics 8.4.204 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14912MiB)
YOLO11s summary (fused): 100 layers, 9,414,348 parameters, 0 gradients, 21.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2346.6±1390.2 MB/s, size: 429.5 KB)
val: Scanning /tmp/roadscan_yolo/labels/val.cache... 5757 images, 2164 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 5757/5757 1.5Git/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 360/360 3.5it/s 1:41
                   all       5757       8447      0.633      0.552      0.596      0.313
                   D00       2053       4083      0.624      0.554      0.587      0.331
                   D10       1178       1832      0.594      0.572      0.589      0.302
                   D20       1255       1561      0.661      0.635      0.683      0.382
                   D40        538        971      0.652      0.449      0.523      0.238
Speed: 0.8ms preprocess, 11.2ms inf

## 5. Export & package

In [9]:
onnx_path = best_model.export(format='onnx', simplify=True, imgsz=IMGSZ)
shutil.copy(BEST_PT, MODELS_DIR / 'best.pt')
shutil.copy(onnx_path, MODELS_DIR / Path(onnx_path).name)
print('Saved best.pt and', Path(onnx_path).name)

metrics_json = {
    'mAP50': float(metrics.box.map50), 'mAP50-95': float(metrics.box.map),
    'precision': float(metrics.box.mp), 'recall': float(metrics.box.mr),
    'per_class': per_class,
}
(MODELS_DIR / 'metrics.json').write_text(json.dumps(metrics_json, indent=2))
print(json.dumps(metrics_json, indent=2))

zip_path = shutil.make_archive(str(WORK_DIR / 'results'), 'zip', root_dir=WORK_DIR, base_dir='models')
print('Done! Download:', zip_path)

Ultralytics 8.4.204 🚀 Python-3.13.16 torch-2.11.0+cu130 CPU (Intel Xeon CPU @ 2.00GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino
YOLO11s summary (fused): 100 layers, 9,414,348 parameters, 0 gradients, 21.4 GFLOPs

PyTorch: starting from '/kaggle/working/roadscan/models/roadscan/weights/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 8, 8400) (18.3 MB)
requirements: Ultralytics requirements ['onnxruntime', 'onnxslim>=0.1.82'] not found, attempting AutoUpdate...
Using Python 3.13.16 environment at: /usr
Resolved 12 packages in 344ms
 Downloaded onnxruntime
Prepared 2 packages in 319ms
Installed 2 packages in 8ms
 + onnxruntime==1.31.1
 + onnxslim==0.1.99

requirements: AutoUpdate success ✅ 2.0s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect


ONNX: starting export with onnx 1.23.2 opset 18...
ONNX: slimming with onnxslim 0.1.99..